<a href="https://colab.research.google.com/github/Meemansha-spec/EDA-with-Pandas/blob/main/Churn_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from overrides.typing_utils import unknown

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
df = pd.read_csv('telecom_customer_churn.csv')

In [14]:
df.head()

,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Number of Referrals,...,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


In [4]:
#find the missing values
df.isna().sum()

,0
Customer ID,0
Gender,0
Age,0
Married,0
Number of Dependents,0
City,0
Zip Code,0
Latitude,0
Longitude,0
Number of Referrals,0


In [15]:
## remove spaces and covert the columns into lowercase
df.columns = df.columns.str.lower().str.replace(r'\s+', '_', regex=True)

In [7]:
df.head()

,customer_id,gender,age,married,number_of_dependents,city,zip_code,latitude,longitude,number_of_referrals,...,payment_method,monthly_charge,total_charges,total_refunds,total_extra_data_charges,total_long_distance_charges,total_revenue,customer_status,churn_category,churn_reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


# Clean missing values.
## Offer
- Missing offer means the customer did not receive any offer

In [16]:
df['offer'] = df['offer'].fillna('NO Offer')

In [18]:
df['offer'].value_counts(dropna=False)

,count
offer,
NO Offer,3877
Offer B,824
Offer E,805
Offer D,602
Offer A,520
Offer C,415


## Phone service related missing values

 - If phone_service = "No", then:
 - multiple_lines should be "No Phone Service"
- avg_monthly_long_distance_charges should be 0

In [27]:
phone_no_mask = df['phone_service'].eq('No')  ## all false values are selected
df.loc[phone_no_mask & df['multiple_lines'].isna(),'multiple_lines'] = 'No Phone Service'  ## select the places where multiple lines are missing where phone service is not there, with no phone service

df.loc[phone_no_mask & df['avg_monthly_long_distance_charges'].isna(),'avg_monthly_long_distance_charges'] = 0.0

#### The missing values were already clean but I added this extra precaution

In [29]:
# Safety handling:
# If phone_service = "Yes" but long distance charge is missing,
# fill with median of customers who have phone service.

phone_yes_mask = df["phone_service"].eq("Yes")

median_long_distance_charge = df.loc[
    phone_yes_mask,
    "avg_monthly_long_distance_charges"
].median()

df.loc[
    phone_yes_mask & df["avg_monthly_long_distance_charges"].isna(),
    "avg_monthly_long_distance_charges"
] = median_long_distance_charge

df.loc[
    phone_yes_mask & df["multiple_lines"].isna(),
    "multiple_lines"
] = "Unknown"

print("Phone-service-related missing values cleaned.")

Phone-service-related missing values cleaned.


In [30]:
# Validate phone-related columns

df[["phone_service", "multiple_lines", "avg_monthly_long_distance_charges"]].isna().sum()

,0
phone_service,0
multiple_lines,0
avg_monthly_long_distance_charges,0


# Internet service related missing values

- if internet service = 'No',
- means customer does not have internet service

In [31]:
internet_no_mask = df['internet_service'].eq('No')
internet_columns = [ "internet_type",
    "online_security", "online_backup",
    "device_protection_plan", "premium_tech_support", "streaming_tv",
    "streaming_movies", "streaming_music", "unlimited_data"]

for col in internet_columns:
    df.loc[internet_no_mask & df[col].isna(), col] = "No Internet Service"


df.loc[internet_no_mask & df["avg_monthly_gb_download"].isna(), "avg_monthly_gb_download"] = 0.0
print("Internet-service-related missing values cleaned for customers without internet service.")

Internet-service-related missing values cleaned for customers without internet service.


In [32]:
df[internet_columns].isna().sum()

,0
internet_type,0
online_security,0
online_backup,0
device_protection_plan,0
premium_tech_support,0
streaming_tv,0
streaming_movies,0
streaming_music,0
unlimited_data,0


In [34]:
df['avg_monthly_gb_download'].isna().sum()

np.int64(0)

## Safety handling

In [36]:
# Safety handling:
# If internet_service = "Yes" but internet-related values are missing,
# mark categorical values as Unknown and numeric GB download with median.

internet_yes_mask = df["internet_service"].eq("Yes")

for col in internet_columns:
    df.loc[
        internet_yes_mask & df[col].isna(),
        col
    ] = "Unknown"

median_gb_download = df.loc[
    internet_yes_mask,
    "avg_monthly_gb_download"
].median()

df.loc[
    internet_yes_mask & df["avg_monthly_gb_download"].isna(),
    "avg_monthly_gb_download"
] = median_gb_download

print("Internet-service-related safety cleaning completed.")

Internet-service-related safety cleaning completed.


## Clean Churn Category and churn reason
- If customr_status is not `churned` , missing churn category means `not_churned`

In [40]:
non_churn_mask = df['customer_status'].ne('Churned')  ## finds all values which are not churned

df.loc[non_churn_mask & df['churn_category'].isna(), 'churn_category'] = 'Not Churned'
df.loc[non_churn_mask & df['churn_reason'].isna(), 'churn_reason'] = 'Not Churned'


In [42]:
df[['customer_status','churn_category']].isna().sum()

,0
customer_status,0
churn_category,0


In [46]:
df.isna().sum()

,0
customer_id,0
gender,0
age,0
married,0
number_of_dependents,0
city,0
zip_code,0
latitude,0
longitude,0
number_of_referrals,0


In [48]:
# Convert ID and zip code to string
# They are identifiers, not mathematical numbers.

df["customer_id"] = df["customer_id"].astype("string")
df["zip_code"] = df["zip_code"].astype("string")

In [49]:
# Convert integer columns

integer_cols = [
    "age",
    "number_of_dependents",
    "number_of_referrals",
    "tenure_in_months",
    "total_extra_data_charges"
]

for col in integer_cols:
    df[col] = df[col].astype("int64")

In [50]:
# Convert float columns

float_cols = [
    "latitude",
    "longitude",
    "avg_monthly_long_distance_charges",
    "avg_monthly_gb_download",
    "monthly_charge",
    "total_charges",
    "total_refunds",
    "total_long_distance_charges",
    "total_revenue"
]

for col in float_cols:
    df[col] = df[col].astype("float64")

In [ ]:
churn_data[columns_to_convert] = churn_data[columns_to_convert].replace({'Yes':1, 'No':0})

/tmp/ipykernel_1262/3175654185.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  churn_data[columns_to_convert] = churn_data[columns_to_convert].replace({'Yes':1, 'No':0})


In [ ]:
#convert male and female to 1 and 0
churn_data['gender'] = churn_data['gender'].replace({'Female':0, 'Male':1})

/tmp/ipykernel_1262/1949619748.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  churn_data['gender'] = churn_data['gender'].replace({'Female':0, 'Male':1})


In [ ]:
## Here is the list of columns which are missing and we have to carefully fill them
missing_value_columns =  ['offer','avg_monthly_long_distance_charges','multiple_lines','internet_type','avg_monthly_gb_download','online_security','online_backup','device_protection_plan','premium_tech_support','streaming_tv','streaming_movies','streaming_music','unlimited_data','churn_category','churn_reason']

### Let's start filling the missing columns

In [ ]:
churn_data['offer'] = churn_data['offer'].fillna('No Offer')

In [ ]:
#where internet service is 0 make all the columns
internet_columns = [
    "online_security", "online_backup",
    "device_protection_plan", "premium_tech_support", "streaming_tv",
    "streaming_movies", "streaming_music", "unlimited_data","internet_service"
]

In [ ]:
churn_data[internet_columns]

,online_security,online_backup,device_protection_plan,premium_tech_support,streaming_tv,streaming_movies,streaming_music,unlimited_data,internet_service
0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1
1,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1
3,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1
4,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1
...,...,...,...,...,...,...,...,...,...
7038,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1
7039,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1
7040,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1
7041,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1


In [ ]:
for col in internet_columns:
    churn_data.loc[churn_data["internet_service"].eq(0) & churn_data[col].isna(), col] = 0

In [ ]:
## select customers who did not churn and marked their mising category as not churned.
churn_data.loc[churn_data["customer_status"].ne("Churned") & churn_data["churn_category"].isna(), "churn_category"] = "Not Churned"
churn_data.loc[churn_data["customer_status"].ne("Churned") & churn_data["churn_reason"].isna(), "churn_reason"] = "Not Churned"

In [ ]:

# If phone_service is 0, customer cannot have multiple lines
churn_data.loc[churn_data["phone_service"] == 0, "multiple_lines"] = 0

# If phone_service is 1, fill only missing values in multiple_lines with mode
mode_multiple_lines = churn_data.loc[churn_data["phone_service"] == 1, "multiple_lines"].mode()[0]

churn_data.loc[
    (churn_data["phone_service"] == 1) & (churn_data["multiple_lines"].isna()),
    "multiple_lines"
] = mode_multiple_lines

# Convert to integer
churn_data["multiple_lines"] = churn_data["multiple_lines"].astype(int)

In [ ]:
# do same for avg long calls
churn_data["avg_monthly_long_distance_charges"] = (
    churn_data.groupby("phone_service")["avg_monthly_long_distance_charges"]
      .transform(lambda x: x.fillna(0) if x.name == 0 else x.fillna(x.median()))
)

In [ ]:
churn_data.isna().sum()

,0
customer_id,0
gender,0
age,0
married,0
number_of_dependents,0
city,0
zip_code,0
latitude,0
longitude,0
number_of_referrals,0


In [ ]:
churn_data["avg_monthly_gb_download"] = (
    churn_data.groupby("internet_service")["avg_monthly_gb_download"]
      .transform(lambda x: x.fillna(0) if x.name == 0 else x.fillna(x.median()))
)

In [ ]:
churn_data['internet_type'] = churn_data['internet_type'].fillna('No Internet Service')

In [ ]:
churn_data.isna().sum()

,0
customer_id,0
gender,0
age,0
married,0
number_of_dependents,0
city,0
zip_code,0
latitude,0
longitude,0
number_of_referrals,0


In [ ]:
## Now the data is clean, now we can upload it on sql for solving the queries
churn_data.to_csv("telecom_customer_churn_cleaned.csv", index=False)

## The Data is cleaned